# Model Comparison Evaluation

## Purpose

This notebook performs comprehensive comparison of all trained models in the SymAD-ECNN project. It supports **dissertation Chapter 8.3 (Model Testing)** and **Chapter 8.4 (Benchmarking)**.

## Inputs

- Result JSON files from trained models stored in Google Drive
- Located in: `/content/drive/MyDrive/symAD-ECNN/results/`

## Outputs

All outputs are saved to Google Drive:
- `/content/drive/MyDrive/symAD-ECNN/evaluations/tables/master_model_results.csv`
- `/content/drive/MyDrive/symAD-ECNN/evaluations/tables/master_model_results.md`
- `/content/drive/MyDrive/symAD-ECNN/evaluations/json/master_model_results.json`
- Various comparison figures in `/content/drive/MyDrive/symAD-ECNN/evaluations/figures/`

## Chapter 8 Alignment

- **8.3 Model Testing**: Presents performance metrics for each model
- **8.4 Benchmarking**: Compares models internally and provides publication-ready tables

---
## 1. Environment Setup

Mount Google Drive and configure paths for Colab execution.

In [ ]:
# Environment setup: mount Google Drive if running in Colab
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Running in Google Colab; Drive mounted at /content/drive")
except ImportError:
    IN_COLAB = False
    print("Running outside Colab; skipping Drive mount")

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime

# Import evaluation utilities
from config import (
    DRIVE_PROJECT_ROOT, RESULTS_DIR, TABLES_DIR, FIGURES_DIR, JSON_DIR,
    ensure_directories_exist, print_config_summary
)
from path_utils import (
    get_drive_project_root, find_results_jsons, validate_paths
)
from metrics_utils import (
    load_results_json, load_all_results_jsons,
    normalize_result_schema, results_to_dataframe,
    rank_models, format_metrics_table
)
from plotting_utils import (
    plot_metric_comparison, plot_radar_comparison,
    save_figure
)
from io_utils import (
    save_json, save_csv, save_markdown_table,
    initialize_output_directories
)

print("All imports successful.")

In [ ]:
# Create output directories
dir_status = ensure_directories_exist()
print("\nDirectory status:")
for name, info in dir_status.items():
    status = "Created" if info.get('created', False) else "Failed"
    print(f"  {name}: {status}")

In [ ]:
# Print configuration summary
print_config_summary()

---
## 2. Path Discovery and Validation

Locate all model result files in Google Drive.

In [ ]:
# Validate project paths
validation = validate_paths(verbose=True)

In [ ]:
# Find all result JSON files
project_root = get_drive_project_root()
results_dict = find_results_jsons(project_root)

print(f"\nResult files found by category:")
for category, files in results_dict.items():
    if files:
        print(f"  {category}: {len(files)} files")
        for f in files[:3]:  # Show first 3
            print(f"    - {f.name}")
        if len(files) > 3:
            print(f"    ... and {len(files) - 3} more")

---
## 3. Load and Parse Result Files

Load all result JSON files and normalize them to a consistent schema.

In [ ]:
# Load all result files
all_result_paths = results_dict.get('all', [])
print(f"Loading {len(all_result_paths)} result files...")

loaded_results = load_all_results_jsons(all_result_paths)
print(f"Successfully loaded: {len(loaded_results)} files")

In [ ]:
# Display raw structure of first result (for debugging)
if loaded_results:
    print("Sample result structure:")
    sample = loaded_results[0]
    print(f"  Source: {sample.get('_source_file', 'unknown')}")
    print(f"  Keys: {list(sample.keys())[:10]}")
    
    # Show nested structure if present
    for nested_key in ['metrics', 'results', 'evaluation']:
        if nested_key in sample and isinstance(sample[nested_key], dict):
            print(f"  {nested_key} keys: {list(sample[nested_key].keys())}")

---
## 4. Build Master Results Table

Create a unified DataFrame with all model results.

In [ ]:
# Import and run the build function
from build_master_results_table import build_master_results_table

master_df, metadata = build_master_results_table(
    root=project_root,
    save_outputs=True,
    verbose=True
)

In [ ]:
# Prepare analysis table (exclude ResNet feature-distance) and display master results
analysis_df = master_df.copy()
if not analysis_df.empty and 'model_name' in analysis_df.columns:
    model_name_lower = analysis_df['model_name'].astype(str).str.lower()
    exclude_mask = model_name_lower.str.contains('resnet') & model_name_lower.str.contains('feature') & model_name_lower.str.contains('distance')
    excluded_count = int(exclude_mask.sum())
    analysis_df = analysis_df.loc[~exclude_mask].reset_index(drop=True)
    if excluded_count > 0:
        print(f"Excluded {excluded_count} ResNet feature-distance row(s) from tables/charts.")

if not analysis_df.empty:
    display_cols = ['model_name', 'auroc', 'auprc', 'accuracy', 'precision', 'recall', 'specificity', 'f1_score']
    display_cols = [c for c in display_cols if c in analysis_df.columns]
    
    print("\n" + "=" * 80)
    print("MASTER MODEL RESULTS TABLE (FILTERED)")
    print("=" * 80)
    display(analysis_df[display_cols])
else:
    print("No results loaded (or all rows were filtered). Please check result files.")

---
## 5. Chapter 8.3: Model Testing Results

This section presents the performance metrics for each trained model, supporting **dissertation section 8.3 (Model Testing)**.

### Interpretation

The table above shows the following metrics for each model:

- **AUROC (Area Under ROC Curve)**: Measures the model's ability to discriminate between normal and anomalous samples across all thresholds. Higher is better (1.0 = perfect, 0.5 = random).
- **AUPRC (Area Under Precision-Recall Curve)**: Focuses on performance for the positive (anomalous) class and is especially informative on imbalanced datasets.

- **Accuracy**: Proportion of correct predictions (both normal and anomaly). Can be misleading with imbalanced datasets.

- **Precision**: Of all samples predicted as anomalies, what proportion are true anomalies? High precision means few false alarms.

- **Recall (Sensitivity)**: Of all true anomalies, what proportion were detected? High recall means few missed anomalies.

- **Specificity**: Of all true normal samples, what proportion were correctly identified as normal? High specificity means few false alarms on normal patients.

- **F1 Score**: Harmonic mean of precision and recall, providing a balanced measure.

In [ ]:
# Generate publication-ready table for Chapter 8.3
if not analysis_df.empty:
    # Select and format columns
    pub_cols = ['model_name', 'auroc', 'auprc', 'accuracy', 'precision', 'recall', 'specificity', 'f1_score']
    pub_cols = [c for c in pub_cols if c in analysis_df.columns]
    
    pub_df = analysis_df[pub_cols].copy()
    pub_df.insert(0, 'Rank', range(1, len(pub_df) + 1))
    
    # Rename for publication
    pub_df = pub_df.rename(columns={
        'model_name': 'Model',
        'auroc': 'AUROC',
        'auprc': 'AUPRC',
        'accuracy': 'Accuracy',
        'precision': 'Precision',
        'recall': 'Recall',
        'specificity': 'Specificity',
        'f1_score': 'F1 Score'
    })
    
    print("\nTable 8.1: Model Performance Summary")
    print("-" * 80)
    display(pub_df)

---
## 6. Chapter 8.4: Benchmarking Analysis

This section provides comparative analysis between models for **dissertation section 8.4 (Benchmarking)**.

### Key Comparisons

1. **Best Overall Model**: Highest AUROC indicates best discrimination ability
2. **Best for Anomaly Detection**: Highest recall minimizes missed anomalies
3. **Best for Specificity**: Highest specificity minimizes false alarms
4. **Best Balanced**: Highest F1 score balances precision and recall

In [ ]:
# Identify best models by different criteria
if not analysis_df.empty:
    print("\n" + "=" * 60)
    print("BENCHMARKING SUMMARY")
    print("=" * 60)
    
    metrics_to_check = {
        'auroc': 'Best Overall (AUROC)',
        'auprc': 'Best Precision-Recall Performance (AUPRC)',
        'recall': 'Best for Anomaly Detection (Recall)',
        'specificity': 'Best for Specificity',
        'f1_score': 'Best Balanced (F1)',
        'accuracy': 'Best Accuracy'
    }
    
    for metric, description in metrics_to_check.items():
        if metric in analysis_df.columns and analysis_df[metric].notna().any():
            best_idx = analysis_df[metric].idxmax()
            best_model = analysis_df.loc[best_idx, 'model_name']
            best_value = analysis_df.loc[best_idx, metric]
            print(f"\n{description}:")
            print(f"  Model: {best_model}")
            print(f"  {metric.upper()}: {best_value:.4f}")

In [ ]:
# Create metric comparison bar chart
if not analysis_df.empty and len(analysis_df) > 1:
    fig, ax = plot_metric_comparison(
        analysis_df,
        metrics=['auroc', 'auprc', 'accuracy', 'precision', 'recall', 'f1_score'],
        model_col='model_name',
        title='Model Performance Comparison',
        figsize=(14, 6),
        save_path=str(FIGURES_DIR / 'model_comparisons' / 'metric_comparison_bar.png')
    )
    plt.show()
else:
    print("Insufficient data for comparison plot.")

In [ ]:
# Create radar chart comparison (if multiple models)
if not analysis_df.empty and len(analysis_df) >= 2:
    # Prepare data for radar chart
    metric_names = ['AUROC', 'AUPRC', 'Accuracy', 'Precision', 'Recall', 'Specificity']
    metric_cols = ['auroc', 'auprc', 'accuracy', 'precision', 'recall', 'specificity']
    
    # Filter to available metrics
    available_metrics = [(n, c) for n, c in zip(metric_names, metric_cols) if c in analysis_df.columns]
    metric_names = [n for n, c in available_metrics]
    metric_cols = [c for n, c in available_metrics]
    
    if len(metric_cols) >= 3:
        # Get top 5 models
        top_models = analysis_df.head(5)
        model_names = top_models['model_name'].tolist()
        metrics_data = [top_models[metric_cols].iloc[i].fillna(0).tolist() for i in range(len(top_models))]
        
        fig, ax = plot_radar_comparison(
            model_names,
            metrics_data,
            metric_names=metric_names,
            title='Top Models Performance Radar',
            save_path=str(FIGURES_DIR / 'model_comparisons' / 'radar_comparison.png')
        )
        plt.show()

---
## 7. Ranked Comparisons

Models ranked by different metrics to support various clinical priorities.

In [ ]:
# Generate ranked tables by each metric
if not analysis_df.empty:
    ranking_metrics = ['auroc', 'auprc', 'recall', 'specificity', 'f1_score']
    
    for metric in ranking_metrics:
        if metric in analysis_df.columns and analysis_df[metric].notna().any():
            ranked = rank_models(analysis_df, metric=metric, ascending=False)
            
            print(f"\n{'=' * 60}")
            print(f"Models Ranked by {metric.upper()}")
            print(f"{'=' * 60}")
            
            display_cols = ['rank', 'model_name', metric]
            display_cols = [c for c in display_cols if c in ranked.columns]
            display(ranked[display_cols].head(10))
            
            # Save ranked table
            save_csv(ranked, f'ranked_by_{metric}.csv')

---
## 8. Statistical Summary

Summary statistics across all models.

In [ ]:
# Compute summary statistics
if not analysis_df.empty:
    numeric_cols = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        stats_df = analysis_df[numeric_cols].describe()
        
        print("\n" + "=" * 60)
        print("METRIC STATISTICS ACROSS FILTERED MODELS")
        print("=" * 60)
        display(stats_df.round(4))
        
        # Save statistics
        save_csv(stats_df.reset_index(), 'metric_statistics.csv')

---
## 9. Save Final Outputs

Ensure all outputs are saved to Google Drive.

In [ ]:
# Final save confirmation
if not analysis_df.empty:
    # Save filtered master table in multiple formats
    csv_path = save_csv(analysis_df, 'master_model_results.csv')
    md_path = save_markdown_table(
        pub_df if 'pub_df' in dir() else analysis_df,
        'master_model_results.md',
        title='Model Comparison Results - Chapter 8'
    )
    json_path = save_json(
        {
            'models': analysis_df.to_dict(orient='records'),
            'metadata': metadata,
            'generated_at': datetime.now().isoformat(),
            'notes': 'ResNet feature-distance rows were excluded from exported comparison outputs.'
        },
        'master_model_results.json'
    )
    
    print("\n" + "=" * 60)
    print("OUTPUTS SAVED")
    print("=" * 60)
    print(f"CSV:      {csv_path}")
    print(f"Markdown: {md_path}")
    print(f"JSON:     {json_path}")

---
## 10. Conclusions for Chapter 8

### Summary of Findings

The model comparison analysis reveals the following key insights:

1. **Best Overall Performance**: [To be filled based on actual results]
   - The model with highest AUROC demonstrates the best overall discrimination ability.

2. **Clinical Trade-offs**:
   - Models with higher recall are better suited for screening applications where missing anomalies is costly.
   - Models with higher specificity are better suited for confirmatory testing where false positives are problematic.

3. **Architecture Comparison**:
   - The ECNN (Equivariant CNN) models leverage rotation invariance which may provide benefits for brain MRI analysis.
   - Baseline CNN autoencoders provide a strong foundation for comparison.

4. **Recommendations**:
   - For deployment, the model selection should consider the specific clinical use case and acceptable error rates.
   - The best balanced model (highest F1) provides a reasonable compromise for general use.

### Next Steps

Further analysis in section 8.5 will examine:
- ECNN threshold optimization and score method comparison
- Detailed error analysis (TP/FP/FN/TN)
- Prototype system testing

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("MODEL COMPARISON EVALUATION COMPLETE")
print("=" * 60)
print(f"\nTotal models compared (filtered): {len(analysis_df) if 'analysis_df' in dir() and not analysis_df.empty else 0}")
if 'master_df' in dir() and not master_df.empty:
    print(f"Total raw models before filtering: {len(master_df)}")
print(f"Output directory: {EVALUATIONS_ROOT}")
print("\nThis notebook supports dissertation sections 8.3 and 8.4.")